# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Vikasbit/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


For the Content Refresh Prioritization lane, one row represents the daily search performance of one webpage for a specific client and date. I will use a mid-panel month, March 2026, for development and verification. The goal is to use information available up to the decision point to prioritize webpages for content refresh.

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


For the Content Refresh Prioritization lane:

FEATURES:
- search_volume — historical search demand available before the refresh decision.
- impressions_90d — recent search visibility available before the decision.
- ctr — historical click-through rate available before the decision.
- avg_position — historical average search position available before the decision.
- word_count — current page content length available before the decision.
- content_age_days — age of the content available before the decision.

LABEL:
- trend_direction — the outcome used to identify whether a page is declining or growing. It is treated as the future outcome and is not used as an input feature.

CONTEXT:
- client_id — identifies the client so the analysis can respect client-level grouping and holdout rules.
- date/month — identifies the observation period and keeps the analysis within the selected March 2026 development window.

EXCLUDED:
- Any private client identifiers, URLs, domains, titles, keywords, or information that is not needed for the refresh-prioritization decision. These are excluded for privacy and because they are not required to answer the research question.

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [2]:
%pip -q install duckdb huggingface_hub

import os
import getpass
import duckdb

HF_TOKEN = os.environ.get("HF_TOKEN")

if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        pass

if not HF_TOKEN:
    HF_TOKEN = getpass.getpass("Paste your Hugging Face READ token: ")

con = duckdb.connect()

con.execute(
    f"CREATE OR REPLACE SECRET hf "
    f"(TYPE huggingface, TOKEN '{HF_TOKEN}')"
)

REL = "hf://datasets/FlyRank/internship-warehouse"

FACT = f"""
read_parquet(
    '{REL}/fact_content_daily_performance/**/*.parquet'
)
"""

print("Warehouse connection ready.")

Warehouse connection ready.


In [4]:
files = con.sql("""
SELECT *
FROM glob('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet')
LIMIT 20
""").df()

print(files.to_string(index=False))

                                                                                                  file
hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2025-01/data_0.parquet
hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2025-02/data_0.parquet
hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2025-03/data_0.parquet
hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2025-04/data_0.parquet
hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2025-05/data_0.parquet
hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2025-06/data_0.parquet
hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2025-07/data_0.parquet
hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2025-08/data_0.parquet
hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance

In [5]:
q1 = con.sql(f"""
SELECT
    COUNT(*) AS rows_checked,
    COUNT(DISTINCT report_date || '|' || client_hash_id || '|' || content_hash_id) AS unique_grain_keys
FROM {FACT}
WHERE month = '2026-03'
""").df()

print(q1)

if q1["rows_checked"].iloc[0] == q1["unique_grain_keys"].iloc[0]:
    print("PASS: one row is uniquely identified by report_date + client_hash_id + content_hash_id.")
else:
    print("CHECK: duplicate grain keys exist.")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   rows_checked  unique_grain_keys
0       9841378            9841378
PASS: one row is uniquely identified by report_date + client_hash_id + content_hash_id.


In [6]:
q2 = con.sql(f"""
SELECT
    COUNT(*) AS march_rows,
    MIN(report_date) AS first_date,
    MAX(report_date) AS last_date
FROM {FACT}
WHERE month = '2026-03'
""").df()

print(q2)

   march_rows first_date  last_date
0     9841378 2026-03-01 2026-03-31


In [10]:
MARCH_FACT = """
read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet'
)
"""

q3 = con.sql(f"""
SELECT
    COUNT(*) AS march_rows,
    COUNT(*) FILTER (
        WHERE gsc_data_available IS TRUE
    ) AS gsc_available_rows,
    COUNT(*) FILTER (
        WHERE ga4_data_available IS TRUE
    ) AS ga4_available_rows
FROM {MARCH_FACT}
""").df()

print(q3)

   march_rows  gsc_available_rows  ga4_available_rows
0     9841378             3611061              413966


In [11]:
features = con.sql(f"""
SELECT
    client_hash_id,
    content_hash_id,
    report_date,
    gsc_impressions,
    gsc_clicks,
    gsc_avg_position,
    ga4_pageviews,
    ga4_sessions
FROM {MARCH_FACT}
WHERE month = '2026-03'
LIMIT 10000
""").df()

print("Feature rows:", len(features))
print("Feature columns:")
print(features.columns.tolist())
features.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Feature rows: 10000
Feature columns:
['client_hash_id', 'content_hash_id', 'report_date', 'gsc_impressions', 'gsc_clicks', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions']


,client_hash_id,content_hash_id,report_date,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_pageviews,ga4_sessions
0,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,2026-03-01,20,0,3.350000,<NA>,<NA>
1,client_73cda7b4e4f265ea,content_05597932fe4da067,2026-03-01,1,0,0.000000,<NA>,<NA>
2,client_73cda7b4e4f265ea,content_7a105f548d9c6916,2026-03-01,125,1,4.928000,<NA>,<NA>
3,client_73cda7b4e4f265ea,content_905aa32a0230694e,2026-03-01,7,0,4.000000,<NA>,<NA>
4,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,2026-03-01,11,0,2.272727,<NA>,<NA>


In [12]:
print("GSC impressions available:", features["gsc_impressions"].notna().sum())
print("GSC clicks available:", features["gsc_clicks"].notna().sum())
print("GSC average position available:", features["gsc_avg_position"].notna().sum())
print("GA4 pageviews available:", features["ga4_pageviews"].notna().sum())
print("GA4 sessions available:", features["ga4_sessions"].notna().sum())

GSC impressions available: 10000
GSC clicks available: 10000
GSC average position available: 8371
GA4 pageviews available: 0
GA4 sessions available: 0


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


This dataset can support observed and directional analysis for content-refresh prioritization, but it cannot establish causal effects. The history is unbalanced across clients and pages, some earlier observations have GSC data without GA4 data, and overlapping time windows can make observations dependent on each other. Therefore, the results should be treated as decision-support rather than proof that a refresh will cause a specific change in search performance.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.